# Fashion metadata EDA

Exploratory analysis of `data/FashionDataset/train/styles_train.csv` using **pandas** and **DuckDB** for tables, and **Plotly** for charts.

**Targets for bivariate analysis:** `subCategory`, `season`, `gender`, `occasion`  
(`occasion` is the dataset `usage` column: Casual / Sports / Ethnic / Formal / …)

`id` is treated as an identifier and is excluded from comparisons. `productDisplayName` is also excluded (near-unique text, not a feature).


## Setup


In [1]:
from pathlib import Path

from IPython.display import display

import duckdb
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

ROOT = Path.cwd().resolve()
if ROOT.name == "eda":
    ROOT = ROOT.parent

DATA_PATH = ROOT / "data" / "FashionDataset" / "train" / "styles_train.csv"

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows", 80)
pd.set_option("display.width", 140)

px.defaults.template = "plotly_white"
px.defaults.color_discrete_sequence = px.colors.qualitative.Set2

print("data:", DATA_PATH)
print("exists:", DATA_PATH.exists())


data: /Users/nhan.ngo/rmit/COSC2753-Project/data/FashionDataset/train/styles_train.csv
exists: True


## Load and clean

Some `productDisplayName` values contain unquoted commas, which pandas reads as extra `Unnamed` columns. Those fragments are folded back into the name, then dropped.


In [ ]:
raw: pd.DataFrame = pd.read_csv(DATA_PATH)
print("raw shape:", raw.shape)
print("raw columns:", list(raw.columns))

df: pd.DataFrame = raw.copy()
unnamed: list[str] = [c for c in df.columns if str(c).startswith("Unnamed")]
for col in unnamed:
    mask: pd.Series = df[col].notna()
    df.loc[mask, "productDisplayName"] = (
        df.loc[mask, "productDisplayName"].fillna("") + df.loc[mask, col].astype(str)
    )
df = df.drop(columns=unnamed)
df = df.rename(columns={"usage": "occasion"})

TARGETS = ["subCategory", "season", "gender", "occasion"]
ID_COLS = ["id"]
TEXT_COLS = ["productDisplayName"]
FEATURE_COLS = [c for c in df.columns if c not in ID_COLS + TEXT_COLS]
COMPARE_COLS = [c for c in FEATURE_COLS if c not in TARGETS]

con: duckdb.DuckDBPyConnection = duckdb.connect()
con.register("styles", df)

print("clean shape:", df.shape)
print("features:", FEATURE_COLS)
print("targets:", TARGETS)
df.head()


raw shape: (38617, 12)
raw columns: ['id', 'gender', 'masterCategory', 'subCategory', 'articleType', 'baseColour', 'season', 'year', 'usage', 'productDisplayName', 'Unnamed: 10', 'Unnamed: 11']
clean shape: (38617, 10)
features: ['gender', 'masterCategory', 'subCategory', 'articleType', 'baseColour', 'season', 'year', 'occasion']
targets: ['subCategory', 'season', 'gender', 'occasion']


,id,gender,masterCategory,subCategory,articleType,baseColour,season,year,occasion,productDisplayName
0,1163,Men,Apparel,Topwear,Tshirts,Blue,Summer,2011,Sports,Nike Sahara Team India Fanwear Round Neck Jersey
1,1164,Men,Apparel,Topwear,Tshirts,Blue,Winter,2015,Sports,Nike Men Blue T20 Indian Cricket Jersey
2,1165,Men,Apparel,Topwear,Tshirts,Blue,Summer,2013,Sports,Nike Mean Team India Cricket Jersey
3,1525,Unisex,Accessories,Bags,Backpacks,Navy Blue,Fall,2010,Casual,Puma Deck Navy Blue Backpack
4,1526,Unisex,Accessories,Bags,Backpacks,Black,Fall,2010,Sports,Puma Big Cat Backpack Black


## Overview


In [ ]:
overview: pd.DataFrame = con.sql(
    """
    SELECT
        COUNT(*) AS n_rows,
        COUNT(DISTINCT id) AS n_ids,
        COUNT(DISTINCT gender) AS n_gender,
        COUNT(DISTINCT masterCategory) AS n_masterCategory,
        COUNT(DISTINCT subCategory) AS n_subCategory,
        COUNT(DISTINCT articleType) AS n_articleType,
        COUNT(DISTINCT baseColour) AS n_baseColour,
        COUNT(DISTINCT season) AS n_season,
        COUNT(DISTINCT year) AS n_year,
        COUNT(DISTINCT occasion) AS n_occasion
    FROM styles
    """
).df()
display(overview)

schema: pd.DataFrame = pd.DataFrame(
    {
        "column": df.columns,
        "dtype": df.dtypes.astype(str).values,
        "n_unique": [df[c].nunique(dropna=True) for c in df.columns],
        "n_missing": df.isna().sum().values,
        "pct_missing": (100 * df.isna().mean()).round(2).values,
    }
)
schema


,n_rows,n_ids,n_gender,n_masterCategory,n_subCategory,n_articleType,n_baseColour,n_season,n_year,n_occasion
0,38617,38617,5,7,41,125,46,4,12,8


,column,dtype,n_unique,n_missing,pct_missing
0,id,int64,38617,0,0.00
1,gender,str,5,0,0.00
2,masterCategory,str,7,0,0.00
3,subCategory,str,41,0,0.00
4,articleType,str,125,0,0.00
5,baseColour,str,46,14,0.04
6,season,str,4,20,0.05
7,year,int64,12,0,0.00
8,occasion,str,8,72,0.19
9,productDisplayName,str,27551,7,0.02


## Missing values


In [7]:
missing: pd.DataFrame = con.sql(
    """
    SELECT column_name, n_missing, ROUND(100.0 * n_missing / n_rows, 2) AS pct_missing
    FROM (
        SELECT
            COUNT(*) AS n_rows,
            COUNT(*) - COUNT(gender) AS gender,
            COUNT(*) - COUNT(masterCategory) AS masterCategory,
            COUNT(*) - COUNT(subCategory) AS subCategory,
            COUNT(*) - COUNT(articleType) AS articleType,
            COUNT(*) - COUNT(baseColour) AS baseColour,
            COUNT(*) - COUNT(season) AS season,
            COUNT(*) - COUNT(year) AS year,
            COUNT(*) - COUNT(occasion) AS occasion,
            COUNT(*) - COUNT(productDisplayName) AS productDisplayName
        FROM styles
    ) s
    UNPIVOT (n_missing FOR column_name IN (
        gender, masterCategory, subCategory, articleType, baseColour,
        season, year, occasion, productDisplayName
    ))
    WHERE n_missing > 0
    ORDER BY n_missing DESC
    """
).df()
display(missing)

fig: go.Figure = px.bar(
    missing,
    x="column_name",
    y="n_missing",
    text="pct_missing",
    title="Missing values by column",
    labels={"column_name": "Column", "n_missing": "Missing rows"},
)
fig.update_traces(texttemplate="%{text}%", textposition="outside")
fig.update_layout(yaxis_title="Missing rows")
fig.show()


,column_name,n_missing,pct_missing
0,occasion,72,0.19
1,season,20,0.05
2,baseColour,14,0.04
3,productDisplayName,7,0.02


## Univariate analysis

Distribution of each metadata field on its own. High-cardinality fields are shown as top-N plus **Other**.


In [8]:
def value_counts_sql(col: str) -> pd.DataFrame:
    return con.sql(
        f"""
        SELECT
            COALESCE(CAST("{col}" AS VARCHAR), 'Missing') AS value,
            COUNT(*) AS n,
            ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct
        FROM styles
        GROUP BY 1
        ORDER BY n DESC
        """
    ).df()


def plot_univariate(col: str, top_n: int | None = None, height: int = 480):
    counts = value_counts_sql(col)
    title = f"{col} distribution"
    if top_n is not None and len(counts) > top_n:
        head = counts.iloc[:top_n].copy()
        other_n = int(counts.iloc[top_n:]["n"].sum())
        other_pct = round(100.0 * other_n / counts["n"].sum(), 2)
        head = pd.concat(
            [head, pd.DataFrame([{"value": "Other", "n": other_n, "pct": other_pct}])],
            ignore_index=True,
        )
        counts = head
        title = f"{col} distribution (top {top_n} + Other)"

    fig = px.bar(
        counts,
        x="n",
        y="value",
        orientation="h",
        text="pct",
        title=title,
        labels={"n": "Count", "value": col},
        height=max(height, 28 * len(counts) + 120),
    )
    fig.update_traces(texttemplate="%{text}%", textposition="outside")
    fig.update_layout(yaxis={"categoryorder": "total ascending"})
    fig.show()
    return counts


In [9]:
print("Low / medium cardinality fields")
for col in ["gender", "masterCategory", "season", "occasion", "year"]:
    display(plot_univariate(col, height=420))


Low / medium cardinality fields


,value,n,pct
0,Men,20918,54.17
1,Women,14160,36.67
2,Unisex,2080,5.39
3,Boys,814,2.11
4,Girls,645,1.67


,value,n,pct
0,Apparel,19147,49.58
1,Accessories,9819,25.43
2,Footwear,8541,22.12
3,Personal Care,1001,2.59
4,Free Items,83,0.21
5,Sporting Goods,25,0.06
6,Home,1,0.00


,value,n,pct
0,Summer,19137,49.56
1,Fall,10512,27.22
2,Winter,7381,19.11
3,Spring,1567,4.06
4,Missing,20,0.05


,value,n,pct
0,Casual,29641,76.76
1,Sports,3940,10.20
2,Ethnic,2570,6.66
3,Formal,2300,5.96
4,Missing,72,0.19
5,Smart Casual,55,0.14
6,Travel,25,0.06
7,Party,13,0.03
8,Home,1,0.00


,value,n,pct
0,2011,13689,35.45
1,2012,13361,34.60
2,2016,5363,13.89
3,2015,2348,6.08
4,2017,1340,3.47
5,2013,1116,2.89
6,2010,805,2.08
7,2018,392,1.02
8,2014,174,0.45
9,2009,20,0.05


In [10]:
print("High cardinality fields")
for col, top_n in [("subCategory", 20), ("articleType", 20), ("baseColour", 20)]:
    display(plot_univariate(col, top_n=top_n, height=560))


High cardinality fields


,value,n,pct
0,Topwear,14409,37.31
1,Shoes,6752,17.48
2,Bags,2648,6.86
3,Bottomwear,2526,6.54
4,Watches,2252,5.83
5,Innerwear,1403,3.63
6,Eyewear,1047,2.71
7,Sandal,927,2.40
8,Fragrance,899,2.33
9,Flip Flops,862,2.23


,value,n,pct
0,Tshirts,6781,17.56
1,Shirts,3089,8.00
2,Casual Shoes,2683,6.95
3,Watches,2252,5.83
4,Sports Shoes,1991,5.16
5,Tops,1614,4.18
6,Kurtas,1556,4.03
7,Handbags,1439,3.73
8,Heels,1048,2.71
9,Sunglasses,1047,2.71


,value,n,pct
0,Black,8814,22.82
1,White,4971,12.87
2,Blue,4441,11.50
3,Brown,3001,7.77
4,Grey,2545,6.59
5,Red,2111,5.47
6,Green,1860,4.82
7,Navy Blue,1680,4.35
8,Pink,1422,3.68
9,Purple,1407,3.64


## Bivariate analysis

Each target is crossed with every other feature (except `id` / `productDisplayName`).

- **Cramér's V** measures association strength between categoricals (0 = independent, 1 = fully associated).
- Charts use **row-normalized %** so class imbalance in the target does not hide mix differences.
- High-cardinality partners are truncated to top-N + Other.


In [11]:
def ident(col: str) -> str:
    return '"' + col.replace('"', '""') + '"'


def cramers_v(a: pd.Series, b: pd.Series) -> float:
    tab = pd.crosstab(a.fillna("Missing"), b.fillna("Missing")).to_numpy(dtype=float)
    n = tab.sum()
    if n == 0 or tab.shape[0] < 2 or tab.shape[1] < 2:
        return np.nan
    expected = tab.sum(axis=1, keepdims=True) * tab.sum(axis=0, keepdims=True) / n
    with np.errstate(divide="ignore", invalid="ignore"):
        chi2 = np.nansum((tab - expected) ** 2 / np.where(expected == 0, np.nan, expected))
    r, k = tab.shape
    phi2 = chi2 / n
    phi2corr = max(0.0, phi2 - ((k - 1) * (r - 1)) / (n - 1))
    rcorr = r - ((r - 1) ** 2) / (n - 1)
    kcorr = k - ((k - 1) ** 2) / (n - 1)
    denom = min(kcorr - 1, rcorr - 1)
    if denom <= 0:
        return np.nan
    return float(np.sqrt(phi2corr / denom))


def association_matrix(columns: list[str]) -> pd.DataFrame:
    mat = pd.DataFrame(index=columns, columns=columns, dtype=float)
    for i, c1 in enumerate(columns):
        mat.loc[c1, c1] = 1.0
        for c2 in columns[i + 1 :]:
            v = cramers_v(df[c1], df[c2])
            mat.loc[c1, c2] = v
            mat.loc[c2, c1] = v
    return mat


def _sql_in_list(series: pd.Series, top_n: int) -> str:
    values = series.fillna("Missing").astype(str).value_counts().head(top_n).index.tolist()
    return ", ".join("'" + v.replace("'", "''") + "'" for v in values)


def _collapse_expr(col: str, top_n: int) -> str:
    expr = f"COALESCE(CAST({ident(col)} AS VARCHAR), 'Missing')"
    n_unique = int(df[col].nunique(dropna=False))
    if n_unique <= top_n + 1:
        return expr
    return (
        f"CASE WHEN {expr} IN ({_sql_in_list(df[col], top_n)}) "
        f"THEN {expr} ELSE 'Other' END"
    )


def bivariate_table(
    target: str,
    feature: str,
    top_n_target: int = 12,
    top_n_feature: int = 15,
) -> pd.DataFrame:
    t_expr = _collapse_expr(target, top_n_target)
    f_expr = _collapse_expr(feature, top_n_feature)
    counts = con.sql(
        f"""
        SELECT
            {t_expr} AS target,
            {f_expr} AS feature,
            COUNT(*) AS n
        FROM styles
        GROUP BY 1, 2
        """
    ).df()
    totals = counts.groupby("target")["n"].transform("sum")
    counts["pct_within_target"] = (100 * counts["n"] / totals).round(2)
    target_order = (
        counts.groupby("target")["n"].sum().sort_values(ascending=False).index.tolist()
    )
    counts["target"] = pd.Categorical(counts["target"], categories=target_order, ordered=True)
    return counts.sort_values(["target", "n"], ascending=[True, False])


def plot_bivariate(
    target: str,
    feature: str,
    top_n_target: int = 12,
    top_n_feature: int = 15,
    height: int = 520,
):
    counts = bivariate_table(
        target,
        feature,
        top_n_target=top_n_target,
        top_n_feature=top_n_feature,
    )
    v = cramers_v(df[target], df[feature])
    t_unique = int(df[target].nunique(dropna=False))
    f_unique = int(df[feature].nunique(dropna=False))
    extras = []
    if t_unique > top_n_target + 1:
        extras.append(f"top {top_n_target} {target} + Other")
    if f_unique > top_n_feature + 1:
        extras.append(f"top {top_n_feature} {feature} + Other")
    suffix = f"  ·  {'; '.join(extras)}" if extras else ""
    title = f"{target} vs {feature}  ·  Cramér's V = {v:.3f}{suffix}"

    fig_bar = px.bar(
        counts,
        x="target",
        y="pct_within_target",
        color="feature",
        title=title,
        labels={
            "target": target,
            "pct_within_target": "% within target",
            "feature": feature,
        },
        height=height,
        category_orders={"target": list(counts["target"].cat.categories)},
    )
    fig_bar.update_layout(barmode="stack", legend_title=feature, xaxis_tickangle=-30)
    fig_bar.show()

    pivot = (
        counts.pivot(index="feature", columns="target", values="pct_within_target")
        .fillna(0)
        .reindex(columns=list(counts["target"].cat.categories))
    )
    fig_hm = px.imshow(
        pivot,
        aspect="auto",
        color_continuous_scale="Blues",
        title=f"{feature} mix within each {target} (% of target class)",
        labels={"color": "% within target"},
        height=max(420, 26 * len(pivot) + 180),
        text_auto=".1f",
    )
    fig_hm.update_xaxes(tickangle=-30)
    fig_hm.show()
    return counts


In [12]:
assoc = association_matrix(FEATURE_COLS)
fig = px.imshow(
    assoc.round(3),
    text_auto=True,
    aspect="auto",
    color_continuous_scale="Teal",
    zmin=0,
    zmax=1,
    title="Cramér's V between metadata features",
    labels={"color": "Cramér's V"},
    height=620,
)
fig.show()

target_assoc = (
    assoc.loc[TARGETS, [c for c in FEATURE_COLS if c not in TARGETS]]
    .round(3)
    .T
    .sort_values(TARGETS[0], ascending=False)
)
print("Association of each target with other features")
target_assoc


Association of each target with other features


,subCategory,season,gender,occasion
masterCategory,0.999,0.471,0.175,0.434
articleType,0.959,0.563,0.515,0.660
year,0.408,0.601,0.180,0.137
baseColour,0.161,0.159,0.210,0.153


### Target: `subCategory`

Compared with `gender`, `masterCategory`, `articleType`, `baseColour`, `season`, `year`, `occasion`.


In [13]:
subcat_partners = [c for c in FEATURE_COLS if c != "subCategory"]
print("Cramér's V vs subCategory")
display(
    pd.Series({c: cramers_v(df["subCategory"], df[c]) for c in subcat_partners}, name="cramers_v")
    .sort_values(ascending=False)
    .to_frame()
)

# Hierarchical view: category tree is the strongest structural signal
tree = con.sql(
    """
    SELECT masterCategory, subCategory, COUNT(*) AS n
    FROM styles
    GROUP BY 1, 2
    ORDER BY n DESC
    """
).df()
fig = px.treemap(
    tree,
    path=["masterCategory", "subCategory"],
    values="n",
    title="subCategory nested in masterCategory",
)
fig.show()

for feature in subcat_partners:
    top_n = 12 if df[feature].nunique(dropna=False) > 12 else 30
    display(plot_bivariate("subCategory", feature, top_n_feature=top_n))


Cramér's V vs subCategory


,cramers_v
masterCategory,0.999332
articleType,0.959103
season,0.533078
occasion,0.465527
year,0.408116
gender,0.302810
baseColour,0.161194


,target,feature,n,pct_within_target
52,Topwear,Men,8524,59.16
3,Topwear,Women,4851,33.67
45,Topwear,Boys,618,4.29
0,Topwear,Girls,342,2.37
4,Topwear,Unisex,74,0.51
36,Shoes,Men,4267,63.20
25,Shoes,Women,2181,32.30
24,Shoes,Unisex,263,3.90
22,Shoes,Boys,27,0.40
42,Shoes,Girls,14,0.21


,target,feature,n,pct_within_target
14,Topwear,Apparel,14409,100.00
7,Shoes,Footwear,6752,100.00
11,Other,Accessories,2253,68.84
2,Other,Apparel,809,24.72
3,Other,Personal Care,102,3.12
6,Other,Free Items,83,2.54
1,Other,Sporting Goods,25,0.76
13,Other,Home,1,0.03
10,Bags,Accessories,2648,100.00
4,Bottomwear,Apparel,2526,100.00


,target,feature,n,pct_within_target
0,Topwear,Tshirts,6780,47.05
16,Topwear,Shirts,3089,21.44
21,Topwear,Tops,1614,11.20
17,Topwear,Kurtas,1556,10.80
1,Topwear,Other,1370,9.51
8,Shoes,Casual Shoes,2683,39.74
7,Shoes,Sports Shoes,1991,29.49
18,Shoes,Heels,1048,15.52
23,Shoes,Other,1028,15.23
12,Shoes,Sandals,2,0.03


,target,feature,n,pct_within_target
66,Topwear,Blue,2322,16.11
58,Topwear,White,2082,14.45
12,Topwear,Black,1924,13.35
103,Topwear,Other,1610,11.17
87,Topwear,Green,1188,8.24
...,...,...,...,...
4,Belts,Navy Blue,15,1.89
72,Belts,Pink,13,1.64
154,Belts,Green,13,1.64
63,Belts,Silver,4,0.50


,target,feature,n,pct_within_target
26,Topwear,Summer,8039,55.79
45,Topwear,Fall,6051,41.99
19,Topwear,Winter,223,1.55
37,Topwear,Spring,96,0.67
12,Shoes,Summer,3060,45.32
6,Shoes,Fall,1832,27.13
4,Shoes,Winter,1591,23.56
16,Shoes,Spring,249,3.69
39,Shoes,Missing,20,0.30
1,Other,Summer,2128,65.02


,target,feature,n,pct_within_target
13,Topwear,2011,8453,58.66
81,Topwear,2012,5233,36.32
5,Topwear,2010,392,2.72
67,Topwear,2013,134,0.93
50,Topwear,2016,133,0.92
...,...,...,...,...
35,Belts,2016,49,6.18
60,Belts,2013,44,5.55
84,Belts,2014,3,0.38
46,Belts,2010,2,0.25


,target,feature,n,pct_within_target
13,Topwear,Casual,10450,72.52
42,Topwear,Ethnic,1962,13.62
38,Topwear,Sports,1079,7.49
16,Topwear,Formal,904,6.27
34,Topwear,Smart Casual,10,0.07
21,Topwear,Missing,3,0.02
12,Topwear,Party,1,0.01
0,Shoes,Casual,4134,61.23
47,Shoes,Sports,1972,29.21
2,Shoes,Formal,612,9.06


### Target: `season`

Compared with `gender`, `masterCategory`, `subCategory`, `articleType`, `baseColour`, `year`, `occasion`.


In [14]:
season_partners = [c for c in FEATURE_COLS if c != "season"]
print("Cramér's V vs season")
display(
    pd.Series({c: cramers_v(df["season"], df[c]) for c in season_partners}, name="cramers_v")
    .sort_values(ascending=False)
    .to_frame()
)

for feature in season_partners:
    top_n = 12 if df[feature].nunique(dropna=False) > 12 else 30
    display(plot_bivariate("season", feature, top_n_feature=top_n))


Cramér's V vs season


,cramers_v
year,0.600964
articleType,0.563325
subCategory,0.533078
masterCategory,0.470964
baseColour,0.158953
occasion,0.135154
gender,0.111958


,target,feature,n,pct_within_target
7,Summer,Men,10082,52.68
4,Summer,Women,6961,36.37
2,Summer,Unisex,892,4.66
6,Summer,Boys,675,3.53
16,Summer,Girls,527,2.75
20,Fall,Men,6942,66.04
12,Fall,Women,2881,27.41
11,Fall,Unisex,495,4.71
18,Fall,Boys,113,1.07
13,Fall,Girls,81,0.77


,target,feature,n,pct_within_target
8,Summer,Apparel,11296,59.03
9,Summer,Footwear,3943,20.60
3,Summer,Accessories,3840,20.07
17,Summer,Free Items,42,0.22
15,Summer,Sporting Goods,12,0.06
10,Summer,Personal Care,4,0.02
18,Fall,Apparel,7147,67.99
19,Fall,Footwear,2384,22.68
11,Fall,Accessories,969,9.22
16,Fall,Sporting Goods,11,0.10


,target,feature,n,pct_within_target
44,Summer,Topwear,8039,42.01
3,Summer,Shoes,3060,15.99
19,Summer,Other,2128,11.12
31,Summer,Bottomwear,1491,7.79
42,Summer,Bags,1329,6.94
25,Summer,Innerwear,1147,5.99
47,Summer,Belts,526,2.75
49,Summer,Sandal,463,2.42
28,Summer,Wallets,436,2.28
22,Summer,Flip Flops,420,2.19


,target,feature,n,pct_within_target
35,Summer,Other,7286,38.07
34,Summer,Tshirts,4114,21.50
43,Summer,Casual Shoes,1331,6.96
1,Summer,Shirts,1258,6.57
41,Summer,Sports Shoes,1169,6.11
30,Summer,Tops,1164,6.08
3,Summer,Kurtas,946,4.94
11,Summer,Handbags,713,3.73
2,Summer,Sandals,436,2.28
44,Summer,Flip Flops,420,2.19


,target,feature,n,pct_within_target
30,Summer,Black,4067,21.25
15,Summer,White,2578,13.47
58,Summer,Other,2467,12.89
10,Summer,Blue,2289,11.96
26,Summer,Grey,1279,6.68
2,Summer,Brown,1254,6.55
32,Summer,Red,1132,5.92
20,Summer,Green,976,5.10
46,Summer,Navy Blue,951,4.97
31,Summer,Pink,780,4.08


,target,feature,n,pct_within_target
22,Summer,2012,11423,59.69
31,Summer,2011,4683,24.47
0,Summer,2016,1570,8.20
24,Summer,2013,648,3.39
14,Summer,2017,337,1.76
37,Summer,2015,247,1.29
8,Summer,2010,94,0.49
36,Summer,2014,86,0.45
6,Summer,2018,46,0.24
41,Summer,2008,2,0.01


,target,feature,n,pct_within_target
2,Summer,Casual,14351,74.99
25,Summer,Sports,2081,10.87
27,Summer,Ethnic,1612,8.42
4,Summer,Formal,1048,5.48
14,Summer,Missing,19,0.10
30,Summer,Travel,11,0.06
19,Summer,Smart Casual,9,0.05
24,Summer,Party,6,0.03
9,Fall,Casual,7037,66.94
16,Fall,Sports,1617,15.38


### Target: `gender`

Compared with `masterCategory`, `subCategory`, `articleType`, `baseColour`, `season`, `year`, `occasion`.


In [15]:
gender_partners = [c for c in FEATURE_COLS if c != "gender"]
print("Cramér's V vs gender")
display(
    pd.Series({c: cramers_v(df["gender"], df[c]) for c in gender_partners}, name="cramers_v")
    .sort_values(ascending=False)
    .to_frame()
)

for feature in gender_partners:
    top_n = 12 if df[feature].nunique(dropna=False) > 12 else 30
    display(plot_bivariate("gender", feature, top_n_feature=top_n))


Cramér's V vs gender


,cramers_v
articleType,0.514933
subCategory,0.302810
baseColour,0.209929
occasion,0.207088
year,0.179699
masterCategory,0.174540
season,0.111958


,target,feature,n,pct_within_target
2,Men,Apparel,10875,51.99
3,Men,Footwear,5480,26.20
6,Men,Accessories,3994,19.09
5,Men,Personal Care,527,2.52
13,Men,Free Items,42,0.20
19,Women,Apparel,6886,48.63
7,Women,Accessories,4334,30.61
20,Women,Footwear,2440,17.23
21,Women,Personal Care,462,3.26
8,Women,Free Items,38,0.27


,target,feature,n,pct_within_target
11,Men,Topwear,8524,40.75
47,Men,Shoes,4267,20.40
22,Men,Bottomwear,1348,6.44
33,Men,Watches,1298,6.21
31,Men,Other,1212,5.79
45,Men,Innerwear,926,4.43
36,Men,Sandal,745,3.56
44,Men,Eyewear,588,2.81
5,Men,Fragrance,524,2.51
13,Men,Belts,521,2.49


,target,feature,n,pct_within_target
35,Men,Other,6305,30.14
34,Men,Tshirts,5085,24.31
1,Men,Shirts,2735,13.07
21,Men,Casual Shoes,2106,10.07
20,Men,Sports Shoes,1548,7.40
39,Men,Watches,1298,6.21
0,Men,Sandals,717,3.43
53,Men,Sunglasses,588,2.81
24,Men,Flip Flops,468,2.24
2,Men,Kurtas,61,0.29


,target,feature,n,pct_within_target
32,Men,Black,5555,26.56
52,Men,White,2941,14.06
42,Men,Blue,2542,12.15
15,Men,Other,1942,9.28
25,Men,Brown,1837,8.78
49,Men,Grey,1724,8.24
11,Men,Navy Blue,1234,5.90
33,Men,Red,1107,5.29
9,Men,Green,872,4.17
58,Men,Purple,511,2.44


,target,feature,n,pct_within_target
5,Men,Summer,10082,48.20
11,Men,Fall,6942,33.19
14,Men,Winter,3010,14.39
2,Men,Spring,877,4.19
18,Men,Missing,7,0.03
4,Women,Summer,6961,49.16
12,Women,Winter,3725,26.31
19,Women,Fall,2881,20.35
16,Women,Spring,586,4.14
21,Women,Missing,7,0.05


,target,feature,n,pct_within_target
3,Men,2011,9015,43.10
27,Men,2012,5789,27.67
34,Men,2016,3452,16.50
0,Men,2013,720,3.44
32,Men,2017,616,2.94
41,Men,2010,563,2.69
24,Men,2018,356,1.70
17,Men,2015,312,1.49
20,Men,2014,72,0.34
42,Men,2009,16,0.08


,target,feature,n,pct_within_target
21,Men,Casual,15691,75.01
2,Men,Sports,2876,13.75
23,Men,Formal,2193,10.48
6,Men,Ethnic,90,0.43
10,Men,Smart Casual,44,0.21
16,Men,Missing,23,0.11
26,Men,Travel,1,0.00
3,Women,Casual,10774,76.09
24,Women,Ethnic,2462,17.39
22,Women,Sports,759,5.36


### Target: `occasion`

`occasion` is `usage` in the raw CSV (Casual, Sports, Ethnic, Formal, …).  
Compared with `gender`, `masterCategory`, `subCategory`, `articleType`, `baseColour`, `season`, `year`.


In [16]:
occasion_partners = [c for c in FEATURE_COLS if c != "occasion"]
print("Cramér's V vs occasion")
display(
    pd.Series({c: cramers_v(df["occasion"], df[c]) for c in occasion_partners}, name="cramers_v")
    .sort_values(ascending=False)
    .to_frame()
)

for feature in occasion_partners:
    top_n = 12 if df[feature].nunique(dropna=False) > 12 else 30
    display(plot_bivariate("occasion", feature, top_n_feature=top_n))


Cramér's V vs occasion


,cramers_v
articleType,0.659644
subCategory,0.465527
masterCategory,0.434160
gender,0.207088
baseColour,0.152567
year,0.137203
season,0.135154


,target,feature,n,pct_within_target
10,Casual,Men,15691,52.94
3,Casual,Women,10774,36.35
2,Casual,Unisex,1758,5.93
17,Casual,Boys,783,2.64
21,Casual,Girls,635,2.14
8,Sports,Men,2876,72.99
25,Sports,Women,759,19.26
24,Sports,Unisex,282,7.16
7,Sports,Boys,21,0.53
19,Sports,Girls,2,0.05


,target,feature,n,pct_within_target
2,Casual,Apparel,14045,47.38
11,Casual,Accessories,8677,29.27
3,Casual,Footwear,5890,19.87
8,Casual,Personal Care,967,3.26
30,Casual,Free Items,58,0.20
0,Casual,Sporting Goods,4,0.01
24,Sports,Footwear,1993,50.58
23,Sports,Apparel,1584,40.20
17,Sports,Accessories,341,8.65
33,Sports,Sporting Goods,21,0.53


,target,feature,n,pct_within_target
25,Casual,Topwear,10450,35.26
50,Casual,Shoes,4134,13.95
18,Casual,Bags,2524,8.52
35,Casual,Other,2188,7.38
37,Casual,Watches,2163,7.30
2,Casual,Bottomwear,1670,5.63
48,Casual,Innerwear,1382,4.66
47,Casual,Eyewear,1039,3.51
23,Casual,Sandal,901,3.04
21,Casual,Fragrance,887,2.99


,target,feature,n,pct_within_target
2,Casual,Other,10003,33.75
3,Casual,Tshirts,5829,19.67
10,Casual,Casual Shoes,2661,8.98
24,Casual,Shirts,2213,7.47
5,Casual,Watches,2163,7.30
17,Casual,Tops,1554,5.24
36,Casual,Handbags,1426,4.81
1,Casual,Sunglasses,1039,3.51
42,Casual,Heels,1017,3.43
11,Casual,Flip Flops,856,2.89


,target,feature,n,pct_within_target
45,Casual,Black,6660,22.47
38,Casual,Other,3739,12.61
27,Casual,Blue,3461,11.68
7,Casual,White,3369,11.37
55,Casual,Brown,2455,8.28
...,...,...,...,...
10,Travel,Pink,1,4.00
67,Party,Black,7,53.85
61,Party,Other,4,30.77
71,Party,Silver,2,15.38


,target,feature,n,pct_within_target
1,Casual,Summer,14351,48.42
19,Casual,Fall,7037,23.74
8,Casual,Winter,6882,23.22
23,Casual,Spring,1371,4.63
18,Sports,Summer,2081,52.82
0,Sports,Fall,1617,41.04
6,Sports,Winter,113,2.87
30,Sports,Spring,109,2.77
16,Sports,Missing,20,0.51
3,Ethnic,Summer,1612,62.72


,target,feature,n,pct_within_target
48,Casual,2012,9744,32.87
4,Casual,2011,9551,32.22
44,Casual,2016,5131,17.31
11,Casual,2015,2226,7.51
52,Casual,2017,1292,4.36
62,Casual,2013,859,2.90
30,Casual,2010,394,1.33
25,Casual,2018,281,0.95
14,Casual,2014,143,0.48
31,Casual,2009,16,0.05


## Quick DuckDB snapshots


In [17]:
print("Top subCategory × occasion cells")
display(
    con.sql(
        """
        SELECT subCategory, occasion, COUNT(*) AS n
        FROM styles
        GROUP BY 1, 2
        ORDER BY n DESC
        LIMIT 15
        """
    ).df()
)

print("Gender mix within masterCategory")
display(
    con.sql(
        """
        SELECT
            masterCategory,
            gender,
            COUNT(*) AS n,
            ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (PARTITION BY masterCategory), 1) AS pct_within_master
        FROM styles
        GROUP BY 1, 2
        ORDER BY masterCategory, n DESC
        """
    ).df()
)

print("Season mix by year")
display(
    con.sql(
        """
        SELECT
            year,
            COALESCE(CAST(season AS VARCHAR), 'Missing') AS season,
            COUNT(*) AS n
        FROM styles
        GROUP BY 1, 2
        ORDER BY year, n DESC
        """
    ).df()
)


Top subCategory × occasion cells


,subCategory,occasion,n
0,Topwear,Casual,10450
1,Shoes,Casual,4134
2,Bags,Casual,2524
3,Watches,Casual,2163
4,Shoes,Sports,1972
5,Topwear,Ethnic,1962
6,Bottomwear,Casual,1670
7,Innerwear,Casual,1382
8,Topwear,Sports,1079
9,Eyewear,Casual,1039


Gender mix within masterCategory


,masterCategory,gender,n,pct_within_master
0,Accessories,Women,4334,44.1
1,Accessories,Men,3994,40.7
2,Accessories,Unisex,1447,14.7
3,Accessories,Girls,27,0.3
4,Accessories,Boys,17,0.2
5,Apparel,Men,10875,56.8
6,Apparel,Women,6886,36.0
7,Apparel,Boys,743,3.9
8,Apparel,Girls,558,2.9
9,Apparel,Unisex,85,0.4


Season mix by year


,year,season,n
0,2007,Spring,2
1,2008,Spring,3
2,2008,Summer,2
3,2008,Winter,1
4,2008,Fall,1
5,2009,Fall,19
6,2009,Summer,1
7,2010,Fall,638
8,2010,Summer,94
9,2010,Winter,70


# Conclusion

Dataset Overview

  - 38,617 rows, each with a unique id — no duplicates
  - 10 columns after cleaning: id, gender, masterCategory, subCategory, articleType, baseColour, season, year, occasion (renamed from usage), productDisplayName
  - Very clean: missing values are negligible — occasion has the most at just 0.19% (72 rows), season 0.05%, baseColour 0.04%

Class Imbalance (key concern for modeling)
| Feature | Dominant class | % | Tail classes |                                                                                                                                    
  | --- | --- | --- | --- |                                                                                                                                                          
  | gender | Men (54%) + Women (37%) | 91% | Boys/Girls/Unisex are tiny minorities |                                                                                                 
  | masterCategory | Apparel (50%) + Accessories (25%) + Footwear (22%) | 97% | Personal Care, Free Items, Sporting Goods, Home are negligible |                                     
  | subCategory | Topwear alone = 37%; top 5 cover ~74% | — | Long tail of 41 categories, many very small |                                                                        
  | season | Summer (50%) + Fall (27%) | 77% | Spring is only 4% |                                                                                                                   
  | occasion | Casual dominates at 77% | — | Sports 10%, Ethnic 7%, Formal 6%; Smart Casual/Travel/Party/Home are near-zero |                                                      
  | year | 2011-2012 account for ~70% of data | — | Very uneven temporal sampling (2007-2009 have <30 rows) |                                                                        
  | baseColour | Black (23%), White (13%), Blue (12%) | 48% | 46 colours total, long tail |   

Feature Associations (Cramer's V)

  The key finding from the association matrix:
| Feature pair | Cramér's V | Interpretation |                                                                                                                                     
  | --- | --- | --- |
  | masterCategory ↔ subCategory | 0.999 | Near-perfect — subCategory is essentially a finer split of masterCategory (hierarchical/redundant) |
  | articleType ↔ subCategory | 0.959 | Also near-redundant — articleType further refines subCategory |
  | articleType ↔ occasion | 0.660 | Strong — article type strongly predicts usage context |
  | year ↔ season | 0.601 | Moderate — different years have different season distributions (data collection artifact) |
  | articleType ↔ season | 0.563 | Moderate — some product types are seasonal |
  | articleType ↔ gender | 0.515 | Moderate — gendered product assortments differ |
  | masterCategory ↔ season/occasion | ~0.43–0.47 | Moderate |
  | baseColour | ≤0.21 with all targets | Weak — colour is relatively independent of other attributes |


Key Takeaways for Modeling

  1. Hierarchical redundancy: masterCategory → subCategory → articleType form a near-deterministic hierarchy. Using all three as features would introduce multicollinearity. Pick one
   level or encode the hierarchy.
  2. Severe class imbalance on all targets, especially occasion (77% Casual). Models will need stratified sampling, class weights, or oversampling for minority classes.
  3. baseColour is the most independent feature — it carries unique signal not captured by category/type, making it potentially valuable as a complementary feature.
  4. Temporal bias: The dataset is concentrated in 2011-2012 (~70%). Year-based features or splits need care — year ↔ season correlation is likely a collection artifact, not a real
  fashion trend signal.
  5. Tiny/rare classes (Home, Smart Casual, Travel, Party, Sporting Goods) may need to be merged or dropped depending on the task.
  6. Missing data is not a problem — all columns have <0.2% missing, so simple imputation or dropping is fine.